# OpenCV Document Validator — Results Explorer

**Two modes:**
1. **Load existing results** — reads `eval_results/batch_results.csv`, re-runs OpenCV to enrich with raw feature scores
2. **Re-run on folder** — point `SCAN_FOLDER` to any directory, run OpenCV fresh, get full score dataframe

Change thresholds in Section 1 and re-run any section independently.

## 1. Config — thresholds

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import dataclasses
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from document_validation import ValidationConfig, validate_document_file_ensemble
from document_validation.validator import validate_document_image, _load_image, _render_pdf_pages
from document_validation.ground_truth import FOLDER_TO_LABEL

# ── Thresholds ──────────────────────────────────────────────────────────────
# Calibrated on test_data_1 (484 files: ConcentForm / Landeclaration / Landrecord)
# is_blur/min → accepted;  is_blur/max → blur
#
# blur_tenengrad_threshold=70  : Pareto-optimal — blur_max recall +16–28pp vs 60,
#   same overall accuracy. blur_max median=76, blur_min median=86 — heavy overlap,
#   no threshold cleanly separates them.
#
# max_document_saturation_p90=999 : saturation gate disabled — phone-photo blur docs
#   share the same saturation range as non-documents; gate at 110 cut blur recall ~25%.
#
# not_document ceiling: is_not_landdeclaration files (40/150) have doc_confidence≈0.935
#   — they ARE real documents of the wrong type; OpenCV cannot distinguish them.
CONFIG = ValidationConfig(
    blur_tenengrad_threshold      = 70.0,
    blur_patch_ratio              = 0.25,
    blur_patch_grid               = 5,
    blur_patch_percentile         = 10.0,
    min_readability_contrast      = 35.0,
    max_low_readability_gray_std  = 65.0,
    min_document_confidence       = 0.60,
    max_document_saturation_p90   = 999.0,   # disabled — see note above
    min_reject_confidence         = 0.75,
    min_cut_confidence            = 0.85,
    pdf_dpi                       = 200,
)

threshold_fields = [
    "blur_tenengrad_threshold", "blur_patch_ratio",
    "blur_patch_grid", "blur_patch_percentile",
    "min_readability_contrast", "max_low_readability_gray_std",
    "min_document_confidence", "max_document_saturation_p90",
    "min_reject_confidence", "min_cut_confidence", "pdf_dpi",
]
cfg_dict = dataclasses.asdict(CONFIG)
display(pd.DataFrame(
    [(k, cfg_dict[k]) for k in threshold_fields],
    columns=["threshold", "value"]
).set_index("threshold"))

PROJECT_ROOT = Path.cwd()
OUT_DIR = PROJECT_ROOT / "eval_results"
OUT_DIR.mkdir(exist_ok=True)
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".pdf"}
LABELS = ["accepted", "blur", "cut", "not_document"]
TIE_BREAK = ("not_document", "blur", "not_clear", "cut", "accepted")

## 2. Helpers — score extraction

In [ ]:
def _label_from_issues(issues):
    normalised = ['blur' if i == 'not_clear' else i for i in issues]
    if not normalised:
        return 'accepted'
    for label in TIE_BREAK:
        if label in normalised:
            return label
    return normalised[0]


def score_page(page_bgr, config=CONFIG):
    """Run OpenCV on one BGR image, return flat score dict."""
    r = validate_document_image(page_bgr, config=config)
    return {
        # blur
        'laplacian_variance'  : round(r.blur.laplacian_variance, 2),
        'tenengrad'           : round(r.blur.tenengrad, 2),
        'readability_contrast': round(r.blur.readability_contrast, 2),
        'grayscale_std'       : round(r.blur.grayscale_std, 2),
        'blur_confidence'     : round(r.blur.confidence, 3),
        # document
        'doc_confidence'      : round(r.document.confidence, 3),
        'page_area_ratio'     : round(r.document.page_area_ratio, 3),
        'ink_ratio'           : round(r.document.ink_ratio, 3),
        'edge_density'        : round(r.document.edge_density, 3),
        # cut
        'cut_confidence'      : round(r.cut.confidence, 3),
        # clear
        'clear_confidence'    : round(r.clear.confidence, 3),
        # thresholds used
        'thresh_laplacian'    : config.blur_laplacian_threshold,
        'thresh_tenengrad'    : config.blur_tenengrad_threshold,
        'thresh_min_doc_conf' : config.min_document_confidence,
        'thresh_min_reject'   : config.min_reject_confidence,
        # prediction
        'raw_issues'          : ', '.join(r.issues) if r.issues else '',
        'predicted_label'     : _label_from_issues(r.issues),
    }


def score_file(file_path, config=CONFIG):
    """Score the first page of a file, return score dict."""
    fp = Path(file_path)
    try:
        if fp.suffix.lower() == '.pdf':
            pages = list(_render_pdf_pages(fp, config))
            page = pages[0] if pages else None
        else:
            page = _load_image(fp)
        if page is None:
            raise ValueError('empty page')
        scores = score_page(page, config)
    except Exception as exc:
        scores = {'predicted_label': 'error', 'raw_issues': str(exc)}
    scores['file_path'] = str(fp)
    scores['filename']  = fp.name
    return scores


print('Helpers loaded.')

## 3. Load existing results

Reads `eval_results/batch_results.csv` (319 files from the last full eval run),
then re-scores each file with the current `CONFIG` to attach raw feature columns.

In [ ]:
BATCH_CSV = OUT_DIR / 'batch_results.csv'

if not BATCH_CSV.exists():
    print(f'batch_results.csv not found at {BATCH_CSV}. Run run_eval.py first.')
else:
    base_df = pd.read_csv(BATCH_CSV)
    base_df['file_path'] = base_df['file_path'].apply(Path)
    print(f'Loaded {len(base_df)} rows from {BATCH_CSV}')
    display(base_df['expected_label'].value_counts().rename('count').to_frame())

In [ ]:
# Re-score with current thresholds to get raw feature columns
# This takes ~5-8 minutes (same as run_eval.py)
# Set RESCORE = False to skip and use only what's in the CSV
RESCORE = True

if RESCORE and BATCH_CSV.exists():
    score_rows = []
    n = len(base_df)
    for i, (_, row) in enumerate(base_df.iterrows()):
        s = score_file(row['file_path'], CONFIG)
        s['expected_label'] = row['expected_label']
        s['doc_type']       = row.get('doc_type', '')
        s['state']          = row.get('state', '')
        score_rows.append(s)
        if (i + 1) % 50 == 0 or i == n - 1:
            print(f'  {i+1}/{n}', end='\r')

    results_df = pd.DataFrame(score_rows)
    results_df.to_csv(OUT_DIR / 'batch_results_scored.csv', index=False)
    print(f'\nRe-scored {len(results_df)} files → eval_results/batch_results_scored.csv')
else:
    # Load from pre-scored CSV if it exists
    scored_csv = OUT_DIR / 'batch_results_scored.csv'
    if scored_csv.exists():
        results_df = pd.read_csv(scored_csv)
        print(f'Loaded pre-scored results: {len(results_df)} rows')
    else:
        results_df = base_df.copy()
        print('Using base CSV (no feature scores — set RESCORE=True to enrich)')

In [ ]:
col_order = [
    'filename', 'doc_type', 'state', 'expected_label', 'predicted_label', 'raw_issues',
    'watermark_blur',
    'laplacian_variance', 'tenengrad', 'readability_contrast', 'grayscale_std',
    'blur_confidence', 'doc_confidence', 'cut_confidence', 'clear_confidence',
    'ink_ratio', 'edge_density', 'page_area_ratio',
    'thresh_laplacian', 'thresh_tenengrad', 'thresh_min_doc_conf', 'thresh_min_reject',
    'file_path',
]
def highlight_wrong(row):
    wrong = row.get('expected_label') != row.get('predicted_label')
    return ['background-color: #ffeaea' if wrong else '' for _ in row]


In [ ]:
# Full results dataframe — sort by expected label then state
results_df = pd.read_csv('eval_results/testdata_full_scored.csv')
show_cols = [c for c in col_order if c in results_df.columns]
display_df = results_df[show_cols].sort_values(['expected_label', 'state'])


display(
    display_df.style
    .apply(highlight_wrong, axis=1)
    .format(precision=3)
)

In [ ]:
len(results_df)

## 4a. Filter by doc type

Set `DOC_TYPE_FILTER` to a list of doc-type folder names to restrict the view.  
Set to `None` to include all three types.

Friendly display names are mapped in `DOC_TYPE_LABELS` — edit as needed.

In [ ]:
# ── Doc-type filter ───────────────────────────────────────────────────────────
# Change DOC_TYPE_FILTER to restrict results. Set to None to show all.
DOC_TYPE_FILTER = ["ConcentForm", "Landeclaration"]   # ← edit here

# Friendly column label for each doc-type folder name
DOC_TYPE_LABELS = {
    "ConcentForm":    "concent form",
    "Landeclaration": "land declaration",
    "Landrecord":     "land record",
}

# ── Apply filter ──────────────────────────────────────────────────────────────
if "doc_type" not in results_df.columns:
    print("WARNING: 'doc_type' column missing — load testdata_full_scored.csv (cell-9) first.")
else:
    if DOC_TYPE_FILTER:
        filtered_df = results_df[results_df["doc_type"].isin(DOC_TYPE_FILTER)].copy()
    else:
        filtered_df = results_df.copy()

    # Add friendly display name as a column
    filtered_df["document type"] = (
        filtered_df["doc_type"].map(DOC_TYPE_LABELS).fillna(filtered_df["doc_type"])
    )

    # Build display column order — put "document type" first, drop raw doc_type
    filter_col_order = ["document type"] + [
        c for c in col_order if c in filtered_df.columns and c != "doc_type"
    ]

    display_filtered = (
        filtered_df[filter_col_order]
        .sort_values(["document type", "expected_label", "state"])
        .reset_index(drop=True)
    )

    shown_types = filtered_df["doc_type"].unique().tolist()
    print(f"Showing {len(filtered_df)} files — doc types: {shown_types}")
    display(
        display_filtered.style
        .apply(highlight_wrong, axis=1)
        .format(precision=3)
    )

In [ ]:
# ── Metrics on filtered set ───────────────────────────────────────────────────
valid_filt = filtered_df[filtered_df["predicted_label"] != "error"].copy()
yt_filt = valid_filt["expected_label"]
yp_filt = valid_filt["predicted_label"]

acc_filt = (yt_filt == yp_filt).mean()
type_label = ", ".join(DOC_TYPE_LABELS.get(t, t) for t in (DOC_TYPE_FILTER or ["all"]))
print(f"Doc types: {type_label}")
print(f"Files: {len(valid_filt)}  |  Accuracy: {acc_filt:.3f}\n")
print(classification_report(yt_filt, yp_filt, labels=LABELS, zero_division=0))

# Confusion matrix
cm_filt = confusion_matrix(yt_filt, yp_filt, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_filt, cmap="Blues")
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted"); ax.set_ylabel("Expected")
ax.set_title(f"Confusion Matrix — {type_label}")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm_filt[i, j]), ha="center", va="center",
                color="white" if cm_filt[i, j] > cm_filt.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

# Per-state breakdown for filtered set
if "state" in valid_filt.columns:
    state_rows_filt = []
    for state, grp in valid_filt.groupby("state"):
        yt, yp = grp["expected_label"], grp["predicted_label"]
        r = {"state": state, "n": len(grp), "accuracy": round((yt == yp).mean(), 3)}
        for cls in LABELS:
            if (yt == cls).any():
                r[f"{cls}_recall"] = round((yp[yt == cls] == cls).mean(), 3)
        state_rows_filt.append(r)
    state_filt_df = (
        pd.DataFrame(state_rows_filt)
        .set_index("state")
        .sort_values("accuracy", ascending=False)
    )
    display(state_filt_df.style.background_gradient(cmap="RdYlGn", subset=["accuracy"]))

## 4. Metrics on existing results

In [ ]:
valid = results_df[results_df['predicted_label'] != 'error'].copy()
y_true, y_pred = valid['expected_label'], valid['predicted_label']

acc = (y_true == y_pred).mean()
print(f'Files: {len(valid)}  |  Accuracy: {acc:.3f}\n')
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Expected')
ax.set_title('Confusion Matrix — existing results')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# Per-state breakdown
if 'state' in valid.columns:
    state_rows = []
    for state, grp in valid.groupby('state'):
        yt, yp = grp['expected_label'], grp['predicted_label']
        r = {'state': state, 'n': len(grp), 'accuracy': round((yt==yp).mean(), 3)}
        for cls in LABELS:
            if (yt==cls).any():
                r[f'{cls}_F1'] = round(f1_score(yt==cls, yp==cls, zero_division=0), 3)
        state_rows.append(r)
    state_df = pd.DataFrame(state_rows).set_index('state').sort_values('accuracy', ascending=False)
    display(state_df.style.background_gradient(cmap='RdYlGn', subset=['accuracy']))

## 4b. Manual-annotation evaluation — test_data_1

Loads the manual review CSV, derives ground-truth labels from the annotator columns,
scores every matched file in `test_data_1` with the current `CONFIG`, and compares.

**Label derivation from manual columns:**
- `manual_proper_document = FALSE` → `not_document`
- `manual_proper_document = TRUE` + `manual_image_clear_and_legible = FALSE` → `blur`
- both TRUE + `manual_is_full_landrecord = FALSE` → `cut`
- all TRUE → `accepted`

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
MANUAL_CSV  = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/manual_data - final_comparison_check_land_record_for_accuracy.csv')
TEST_DATA_1 = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1')
MANUAL_OUT  = OUT_DIR / 'manual_eval_scored.csv'

# ── Doc types to evaluate in 4b ───────────────────────────────────────────────
SECTION_4B_DOC_TYPES = ["ConcentForm", "Landeclaration"]   # ← edit here

# ── Load manual annotations ──────────────────────────────────────────────────
ann = pd.read_csv(MANUAL_CSV)

def derive_manual_label(row):
    if not row['manual_proper_document']:
        return 'not_document'
    if not row['manual_image_clear_and_legible']:
        return 'blur'
    if not row['manual_is_full_landrecord']:
        return 'cut'
    return 'accepted'

ann['manual_label'] = ann.apply(derive_manual_label, axis=1)
print(f'Manual annotation CSV: {len(ann)} rows')
print('Ground-truth distribution (full CSV):')
display(ann['manual_label'].value_counts().rename('count').to_frame())

# ── Build file index from ConcentForm + Landeclaration folders only ────────────
file_index = {}
for doc_type in SECTION_4B_DOC_TYPES:
    doc_dir = TEST_DATA_1 / doc_type
    if doc_dir.exists():
        for f in sorted(doc_dir.rglob('*')):
            if f.is_file() and f.suffix.lower() in EXTENSIONS:
                file_index[f.name] = f
    else:
        print(f'WARNING: {doc_dir} not found')

# Filter manual CSV to only files in those folders
matched = ann[ann['file_name'].isin(file_index)]
missing = ann[~ann['file_name'].isin(file_index)]
print(f'\nDoc types scanned: {SECTION_4B_DOC_TYPES}')
print(f'Files indexed: {len(file_index)}')
print(f'Manual CSV rows matched: {len(matched)} / {len(ann)}')

In [ ]:
# ── Score all matched files ───────────────────────────────────────────────────
meta_cols = [c for c in ['state', 'doc_type', 'manual_land_record', 'manual_phone_photo'] if c in ann.columns]

manual_rows = []
n = len(matched)
for i, (_, row) in enumerate(matched.iterrows()):
    fpath = file_index[row['file_name']]
    s = score_file(fpath, CONFIG)
    s['manual_label'] = row['manual_label']
    for col in meta_cols:
        s[col] = row[col]
    manual_rows.append(s)
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f'  {i+1}/{n}', end='\r')

manual_df = pd.DataFrame(manual_rows)

# ── is_blur/min → "accepted" override ─────────────────────────────────────────
# Files physically under test_data_1/.../is_blur/min/ are minimally blurry
# and treated as accepted, regardless of the manual CSV annotation.
def _is_blur_min(fp: str) -> bool:
    parts = Path(fp).parts
    for i in range(len(parts) - 1):
        if parts[i] == 'is_blur' and parts[i + 1] == 'min':
            return True
    return False

blur_min_mask = manual_df['file_path'].apply(_is_blur_min)
manual_df.loc[blur_min_mask, 'manual_label'] = 'accepted'
print(f'\nis_blur/min → accepted: {blur_min_mask.sum()} files overridden')

# ── Derive doc_type from file path if not in manual CSV ───────────────────────
if 'doc_type' not in manual_df.columns:
    manual_df['doc_type'] = manual_df['file_path'].apply(
        lambda fp: Path(fp).relative_to(TEST_DATA_1).parts[0]
        if TEST_DATA_1 in Path(fp).parents else None
    )

manual_df.to_csv(MANUAL_OUT, index=False)
print(f'Done — {len(manual_df)} files → {MANUAL_OUT}')
print('\nGround-truth distribution after is_blur/min override:')
display(manual_df['manual_label'].value_counts().rename('count').to_frame())
if 'doc_type' in manual_df.columns:
    print('\ndoc_type distribution:')
    display(manual_df['doc_type'].value_counts().rename('count').to_frame())

In [ ]:
# ── Filter + display — wrong rows highlighted red ─────────────────────────────
# Uses DOC_TYPE_FILTER / DOC_TYPE_LABELS from section 4a (run that cell first).
# Set DOC_TYPE_FILTER = None there to show all doc types.

manual_col_order = [
    'filename', 'manual_label', 'predicted_label', 'raw_issues',
    'watermark_blur',
    'laplacian_variance', 'tenengrad', 'readability_contrast', 'grayscale_std',
    'blur_confidence', 'doc_confidence', 'cut_confidence', 'clear_confidence',
    'ink_ratio', 'edge_density', 'page_area_ratio',
    'file_path',
]

def highlight_wrong_manual(row):
    wrong = row.get('manual_label') != row.get('predicted_label')
    return ['background-color: #ffeaea' if wrong else '' for _ in row]

# Apply doc-type filter when column is present
if "doc_type" in manual_df.columns and DOC_TYPE_FILTER:
    display_manual = manual_df[manual_df["doc_type"].isin(DOC_TYPE_FILTER)].copy()
else:
    display_manual = manual_df.copy()

# Add friendly "document type" column
if "doc_type" in display_manual.columns:
    display_manual["document type"] = (
        display_manual["doc_type"].map(DOC_TYPE_LABELS).fillna(display_manual["doc_type"])
    )
    sort_cols   = ["document type", "manual_label"]
    manual_show = ["document type"] + [c for c in manual_col_order if c in display_manual.columns]
else:
    sort_cols   = ["manual_label", "filename"]
    manual_show = [c for c in manual_col_order if c in display_manual.columns]

display_manual = display_manual.sort_values(sort_cols).reset_index(drop=True)

shown = display_manual["doc_type"].unique().tolist() if "doc_type" in display_manual.columns else ["all"]
print(f"Showing {len(display_manual)} files — doc types: {shown}")
display(
    display_manual[manual_show].style
    .apply(highlight_wrong_manual, axis=1)
    .format(precision=3)
)

In [ ]:
display_manual[(display_manual['manual_label'] == 'blur') & (display_manual['predicted_label'] == 'accepted')]

In [ ]:
display_manual.columns

In [ ]:
display_manual[display_manual['filename'].str.startswith('119361_Farm_126428_landrec.pdf')]['file_path'].iloc[0]

In [ ]:
# ── Metrics on filtered 4b set ────────────────────────────────────────────────
# display_manual is already filtered to DOC_TYPE_FILTER (set in section 4a)
valid_m = display_manual[display_manual['predicted_label'] != 'error'].copy()
yt_m, yp_m = valid_m['manual_label'], valid_m['predicted_label']

type_label_m = (
    ", ".join(DOC_TYPE_LABELS.get(t, t) for t in (DOC_TYPE_FILTER or []))
    if DOC_TYPE_FILTER else "all doc types"
)
acc_m = (yt_m == yp_m).mean()
print(f"Doc types: {type_label_m}")
print(f'Files: {len(valid_m)}  |  Accuracy: {acc_m:.3f}\n')
print(classification_report(yt_m, yp_m, labels=LABELS, zero_division=0))

# Confusion matrix
cm_m = confusion_matrix(yt_m, yp_m, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_m, cmap='Blues')
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Manual label')
ax.set_title(f'Confusion Matrix — 4b: {type_label_m}')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm_m[i, j]), ha='center', va='center',
                color='white' if cm_m[i, j] > cm_m.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

# Per-state breakdown
if 'state' in valid_m.columns:
    state_rows_m = []
    for state, grp in valid_m.groupby('state'):
        yt, yp = grp['manual_label'], grp['predicted_label']
        r = {'state': state, 'n': len(grp), 'accuracy': round((yt == yp).mean(), 3)}
        for cls in LABELS:
            if (yt == cls).any():
                r[f'{cls}_recall'] = round((yp[yt == cls] == cls).mean(), 3)
        state_rows_m.append(r)
    state_df_m = (
        pd.DataFrame(state_rows_m)
        .set_index('state')
        .sort_values('accuracy', ascending=False)
    )
    display(state_df_m.style.background_gradient(cmap='RdYlGn', subset=['accuracy']))

## 5. Re-run on a specific folder

Point `SCAN_FOLDER` to any directory.  
- If the folder follows the `doc_type/state/category/` layout, ground truth labels are inferred automatically.  
- Otherwise all files are scored with `expected_label = 'unknown'`.

Results are saved to `eval_results/folder_run_<folder_name>.csv`.

In [ ]:
# ── Set the folder to scan ───────────────────────────────────────────────────
SCAN_FOLDER = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear')   # ← change to any path

# ── Run ─────────────────────────────────────────────────────────────────────
assert SCAN_FOLDER.exists(), f'Folder not found: {SCAN_FOLDER}'

# Collect files and infer labels where possible
file_entries = []
for f in sorted(SCAN_FOLDER.rglob('*')):
    if not f.is_file() or f.suffix.lower() not in EXTENSIONS:
        continue
    # Try to infer label from parent folder name
    label = FOLDER_TO_LABEL.get(f.parent.name, 'unknown')
    file_entries.append({'file_path': f, 'expected_label': label})

print(f'Found {len(file_entries)} files in {SCAN_FOLDER}')
label_counts = pd.Series([e['expected_label'] for e in file_entries]).value_counts()
display(label_counts.rename('count').to_frame())

In [ ]:
# Score all files
folder_rows = []
n = len(file_entries)
for i, entry in enumerate(file_entries):
    s = score_file(entry['file_path'], CONFIG)
    s['expected_label'] = entry['expected_label']
    folder_rows.append(s)
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f'  {i+1}/{n}', end='\r')

folder_df = pd.DataFrame(folder_rows)
out_csv = OUT_DIR / f'folder_run_{SCAN_FOLDER.name}.csv'
folder_df.to_csv(out_csv, index=False)
print(f'\nDone. {len(folder_df)} files → {out_csv}')

In [ ]:
# Display results
show_cols_f = [c for c in col_order if c in folder_df.columns]
folder_display = folder_df[show_cols_f].sort_values(['expected_label', 'filename'] if 'expected_label' in folder_df.columns else ['filename'])

display(
    folder_display.style
    .apply(highlight_wrong, axis=1)
    .format(precision=3)
)

In [ ]:
# Metrics — only if ground truth labels are available
known = folder_df[folder_df['expected_label'] != 'unknown']
if len(known) == 0:
    print('No ground-truth labels found — cannot compute metrics.')
    print('Prediction distribution:')
    display(folder_df['predicted_label'].value_counts().rename('count').to_frame())
else:
    valid_f = known[known['predicted_label'] != 'error']
    yt, yp = valid_f['expected_label'], valid_f['predicted_label']
    print(f'Files with labels: {len(valid_f)}  |  Accuracy: {(yt==yp).mean():.3f}\n')
    print(classification_report(yt, yp, labels=LABELS, zero_division=0))

    cm = confusion_matrix(yt, yp, labels=LABELS)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Expected')
    ax.set_title(f'Confusion Matrix — {SCAN_FOLDER.name}')
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.colorbar(im); plt.tight_layout(); plt.show()

## 6. Feature score distributions (optional)

Shows how Laplacian variance, Tenengrad, and document confidence separate by class — useful for threshold tuning.

In [ ]:
# Use whichever df has feature scores
plot_df = results_df if 'laplacian_variance' in results_df.columns else folder_df
plot_df = plot_df[plot_df['expected_label'].isin(LABELS)].copy()

score_cols = ['laplacian_variance', 'tenengrad', 'blur_confidence', 'doc_confidence',
              'cut_confidence', 'clear_confidence']
score_cols = [c for c in score_cols if c in plot_df.columns]

COLORS = {'accepted': '#2ecc71', 'blur': '#e74c3c', 'cut': '#f39c12', 'not_document': '#9b59b6'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, score_cols):
    for label in LABELS:
        vals = plot_df[plot_df['expected_label'] == label][col].dropna()
        if len(vals):
            ax.hist(vals, bins=30, alpha=0.5, label=label, color=COLORS.get(label, 'gray'))
    ax.set_title(col)
    ax.set_xlabel('value'); ax.set_ylabel('count')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Hide unused axes
for ax in axes[len(score_cols):]:
    ax.set_visible(False)

plt.suptitle('Feature score distributions by class', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np

# 1. Load image in grayscale
img = cv2.imread('/Users/mipl/Documents/Screenshot 2026-06-24 at 9.50.06 AM.png', cv2.IMREAD_GRAYSCALE)

# 2. Compute Sobel gradients (First Derivatives)
sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

# 3. Calculate Squared Gradient Magnitude
grad_magnitude_sq = (sobel_x ** 2) + (sobel_y ** 2)

# 4. Take the square root to get the actual magnitude map
grad_magnitude = np.sqrt(grad_magnitude_sq)

# 5. Normalize to 0-255 so it can be rendered as a visual image
tenengrad_visual = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

# 6. Calculate the final score (sum of all pixels)
tenengrad_score = np.sum(grad_magnitude_sq)
print(f"Tenengrad Sharpness Score: {tenengrad_score}")

# Display the visualization
cv2.imshow("What Tenengrad Sees", tenengrad_visual)
cv2.waitKey(0)
cv2.destroyAllWindows()

Tenengrad Sharpness Score: 5608601568.0
